In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **Convertor set de date format BIO → JSON pentru antrenare NER**



Fișier sursă: `NUMERAL.txt`

Entitate țintă: NUMERAL  (etichete B-NUMERAL / I-NUMERAL)


## Particularități gestionate:


*   subtokeni cu prefix ##  → recompuși în cuvinte întregi înainte de export
*   tokenizare cu cratime    → tratate ca tokeni separați cu etichetă O
*   secvențe BIO multi-token → reunite în span-uri cu offset de caracter

## Format de ieșire:


*   `ner_token_level.json`  — liste de tokeni + etichete BIO
                             (compatibil Hugging Face datasets / spaCy)


In [ ]:
import json
import re
from pathlib import Path


# ── Configuration ─────────────────────────────────────────────────────────────

INPUT_FILE = Path("/content/drive/MyDrive/CERC_2026/NUMERAL.txt")
OUT_TOKEN  = Path("ner_token_level.json")

# Strict set of valid labels according to the project's BIO scheme
VALID_TAGS = {"B-NUMERAL", "I-NUMERAL", "O"}


# ── 1. CoNLL File Parsing ─────────────────────────────────────────────────────

def parse_dataset(filepath: Path) -> list[dict]:
    sentences: list[dict] = []
    current_id: int | None = None
    current_tokens: list[tuple[str, str]] = []

    with open(filepath, encoding="utf-8") as fh:
        for line in fh:
            # Strip \r\n — the source file uses Windows line endings
            line = line.rstrip("\r\n")

            # Identifier line: # id = N
            m = re.match(r"^#\s*id\s*=\s*(\d+)$", line.strip())
            if m:
                if current_id is not None:
                    sentences.append({"id": current_id, "raw_tokens": current_tokens})
                current_id = int(m.group(1))
                current_tokens = []
                continue

            # Empty line: sentence separator — skip
            if not line.strip():
                continue

            # Token line:
            #   label = last column  (parts[-1])
            #   token = everything before the last column, rejoined with a space
            parts = line.split()
            if len(parts) >= 2:
                label = parts[-1]
                token = " ".join(parts[:-1])

                # Extra guard: if the label is not in the valid set,
                # the line has an unexpected structure — skip it
                if label not in VALID_TAGS:
                    continue

                current_tokens.append((token, label))

    # Save the last sentence
    if current_id is not None and current_tokens:
        sentences.append({"id": current_id, "raw_tokens": current_tokens})

    return sentences


# ── 2. WordPiece Subtoken Recomposition (##) ──────────────────────────────────

def merge_subtokens(raw_tokens: list[tuple[str, str]]) -> list[tuple[str, str]]:
    """
    Merges subtokens with the ## prefix into the preceding token.
    The label of the merged token is inherited from the first subtoken in the group.

    Examples:
        [("salva","O"), ("##mont","O"), ("##iști","O")]
          →  [("salvamontisti", "O")]

        [("doctor","I-NUMERAL"), ("##and","I-NUMERAL")]
          →  [("doctorand", "I-NUMERAL")]
    """
    merged: list[tuple[str, str]] = []
    for word, label in raw_tokens:
        if word.startswith("##") and merged:
            prev_word, prev_label = merged[-1]
            merged[-1] = (prev_word + word[2:], prev_label)
        else:
            merged.append((word, label))
    return merged


# ── 3. Build Token-Level JSON Record ──────────────────────────────────────────

def make_token_record(sentence: dict) -> dict:
    tokens = merge_subtokens(sentence["raw_tokens"])
    return {
        "id":       sentence["id"],
        "tokens":   [w for w, _ in tokens],
        "ner_tags": [l for _, l in tokens],
    }


# ── 4. Statistics ─────────────────────────────────────────────────────────────

def print_stats(records: list[dict]) -> None:
    total_entities = sum(
        sum(1 for t in r["ner_tags"] if t == "B-NUMERAL")
        for r in records
    )
    tag_counts: dict[str, int] = {}
    for r in records:
        for t in r["ner_tags"]:
            tag_counts[t] = tag_counts.get(t, 0) + 1

    print(f"\n{'─'*50}")
    print(f"  Sentences processed : {len(records)}")
    print(f"  NUMERAL entities    : {total_entities}")
    print(f"  Label distribution  :")
    for tag, cnt in sorted(tag_counts.items()):
        print(f"      {tag:<15} {cnt}")
    print(f"{'─'*50}\n")


# ── 5. Main Pipeline ──────────────────────────────────────────────────────────

def main() -> None:
    print(f"\n[1/3] Reading source file: {INPUT_FILE}")
    sentences = parse_dataset(INPUT_FILE)
    print(f"      → {len(sentences)} sentences found")

    print("[2/3] Building token-level records ...")
    records = [make_token_record(s) for s in sentences]

    print(f"[3/3] Writing JSON output: {OUT_TOKEN}")
    with open(OUT_TOKEN, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"      ✓ {len(records)} records written")

    print_stats(records)

    # Validation: ensure no invalid labels slipped through
    invalid_found = [
        (r["id"], tok, tag)
        for r in records
        for tok, tag in zip(r["tokens"], r["ner_tags"])
        if tag not in VALID_TAGS
    ]
    if invalid_found:
        print(f"⚠️  WARNING: {len(invalid_found)} invalid labels detected:")
        for rid, tok, tag in invalid_found[:10]:
            print(f"   id={rid}  token={repr(tok)}  tag={repr(tag)}")
    else:
        print("✓  All labels are valid — no sanitization needed.")

    # Preview first record
    print("\n── Preview (id=1) ──────────────────────────────────────────────")
    print(json.dumps(records[0], ensure_ascii=False, indent=2))

    print("\nConversion completed successfully! ✓")


if __name__ == "__main__":
    main()

In [ ]:
import shutil
import os

drive_path = '/content/drive/MyDrive/CERC_2026'

# Copy the file to Drive
shutil.copy('ner_token_level.json', f'{drive_path}/ner_token_level.json')
print(f"The dataset has been saved in Drive: {drive_path}")

# **Instalarea bibliotecilor necesare**

In [ ]:
!pip install -q transformers datasets evaluate accelerate seqeval

# **Încărcarea datelor și pregătirea pentru BERT**

## Următorul cod va face trei lucruri esențiale:

1. Va încărca fișierul ner_token_level.json.

2. Va încărca Tokenizer-ul specific pentru bert-base-romanian-cased-v1.

3. Va alinia etichetele: deoarece BERT sparge cuvintele în bucățele (WordPiece), trebuie să ne asigurăm că eticheta B-NUMERAL rămâne la prima bucățică, iar restul sunt ignorate (folosind valoarea -100).

In [ ]:
import json
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, DataCollatorForTokenClassification, AutoModelForTokenClassification, TrainingArguments, Trainer
import numpy as np

# 1. Load the cleaned data
with open('/content/drive/MyDrive/CERC_2026/ner_token_level.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Define label mapping
label_list = ["O", "B-NUMERAL", "I-NUMERAL"]
label_to_id = {label: i for i, label in enumerate(label_list)}

# Transform text labels into numerical IDs
for item in raw_data:
    item['ner_tags'] = [label_to_id[tag] for tag in item['ner_tags']]

# Create an official Hugging Face Dataset
full_dataset = Dataset.from_list(raw_data)
# Split: 80% training, 20% testing
ds = full_dataset.train_test_split(test_size=0.2)

# 2. Load the Tokenizer
model_checkpoint = "dumitrescustefan/bert-base-romanian-cased-v1"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100) # Ignore subsequent sub-tokens
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Process the entire dataset
tokenized_ds = ds.map(tokenize_and_align_labels, batched=True)

# **Configurarea și începerea antrenării**

In [ ]:
# 1. Reload the model (to ensure we start from scratch)
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list)
)

# 2. Configure training arguments (No logging_dir to avoid warnings)
training_args = TrainingArguments(
    output_dir="./results_numeral_ner",
    eval_strategy="epoch",            # Evaluation at each epoch
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# 3. Initialize the Trainer (Corrected: processing_class instead of tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    processing_class=tokenizer,      # This is the change that resolves the error
    data_collator=DataCollatorForTokenClassification(tokenizer),
)

# 4. START TRAINING
print("Starting the fine-tuning process on the dataset...")
trainer.train()

In [ ]:
# Define the label mapping
id2label = {0: "O", 1: "B-NUMERAL", 2: "I-NUMERAL"}
label2id = {"O": 0, "B-NUMERAL": 1, "I-NUMERAL": 2}

# Inject labels into the model configuration BEFORE saving
model.config.id2label = id2label
model.config.label2id = label2id

# Save the model, configuration, and tokenizer in the Drive folder
model_save_path = '/content/drive/MyDrive/CERC_2026/model_final'

trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"The model has been permanently saved in: {model_save_path}")